# 📊 NHPP Software Reliability Models - Complete Analysis

**Based on Chapter 6: System Software Reliability by Hoang Pham**

This notebook demonstrates all 13+ NHPP models for software reliability analysis.

---

## 📋 Table of Contents
1. [Setup & Installation](#setup)
2. [Load Core Implementation](#implementation)
3. [Generate/Load Sample Data](#data)
4. [Exploratory Analysis](#explore)
5. [Fit Individual Models](#fit)
6. [Compare All Models](#compare)
7. [Model Validation](#validate)
8. [Make Predictions](#predict)
9. [Visualizations](#visualize)
10. [Advanced Analysis](#advanced)

<a id='setup'></a>
## 1️⃣ Setup & Installation

Install required packages:

In [ ]:
# Install dependencies (if needed)
!pip install numpy pandas scipy matplotlib -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import integrate, stats
from scipy.optimize import minimize, differential_evolution
import warnings
warnings.filterwarnings('ignore')

print("✅ All packages imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

: 

<a id='implementation'></a>
## 2️⃣ Core NHPP Models Implementation

Let's implement all the models:

In [ ]:
# ============================================================================
# CORE NHPP MODEL CLASS
# ============================================================================

class NHPPModel:
    """
    Generic NHPP model wrapper for software reliability analysis.
    """
    def __init__(self, param_names, m_func, lambda_func, initial_guess=None, bounds=None):
        self.param_names = list(param_names)
        self.m_func = m_func
        self.lambda_func = lambda_func
        self.initial_guess = initial_guess if initial_guess is not None else [1.0]*len(param_names)
        self.bounds = bounds
        self.fitted_params = None
        self.fit_result = None

    def loglik_type2(self, failure_times, p, T=None):
        """Type-II log-likelihood (failure times)"""
        failure_times = np.asarray(failure_times, dtype=float)
        if failure_times.size == 0:
            return -np.inf
        if T is None:
            T = float(failure_times[-1])
        
        try:
            lambdas = np.array([self.lambda_func(t, p) for t in failure_times], dtype=float)
            if np.any(lambdas <= 0) or np.any(np.isnan(lambdas)):
                return -1e99
            m_T = float(self.m_func(T, p))
            if np.isnan(m_T) or np.isinf(m_T):
                return -1e99
            logL = np.sum(np.log(lambdas)) - m_T
            return logL
        except:
            return -1e99

    def fit_type2(self, failure_times, T=None, method='L-BFGS-B', use_de=False):
        """Fit to failure times via MLE"""
        x0 = np.array(self.initial_guess, dtype=float)
        
        if use_de and self.bounds is not None:
            res = differential_evolution(
                lambda x: -self.loglik_type2(failure_times, x, T),
                bounds=self.bounds, seed=42, maxiter=1000
            )
        else:
            res = minimize(
                lambda x: -self.loglik_type2(failure_times, x, T),
                x0, method=method, bounds=self.bounds
            )
        
        self.fitted_params = res.x
        self.fit_result = {
            'params': res.x,
            'loglik': -res.fun,
            'success': res.success,
            'message': str(res.message) if hasattr(res, 'message') else 'OK'
        }
        return self.fit_result

    def predict_failures(self, t, params=None):
        """Predict cumulative failures at time t"""
        if params is None:
            params = self.fitted_params
        return self.m_func(t, params)

    def predict_intensity(self, t, params=None):
        """Predict failure intensity at time t"""
        if params is None:
            params = self.fitted_params
        return self.lambda_func(t, params)

print("✅ NHPPModel class defined!")

: 

In [ ]:
# ============================================================================
# MODEL IMPLEMENTATIONS
# ============================================================================

# 1. Goel-Okumoto Model
def goel_okumoto_m(t, p):
    a, b = p[0], p[1]
    return a * (1.0 - np.exp(-b * t))

def goel_okumoto_lambda(t, p):
    a, b = p[0], p[1]
    return a * b * np.exp(-b * t)

GoelOkumoto = NHPPModel(
    param_names=['a', 'b'],
    m_func=goel_okumoto_m,
    lambda_func=goel_okumoto_lambda,
    initial_guess=[100.0, 0.1],
    bounds=[(1e-8, 1e6), (1e-12, 10)]
)

# 2. Delayed S-shaped Model
def delayed_s_m(t, p):
    a, b = p[0], p[1]
    return a * (1.0 - (1.0 + b * t) * np.exp(-b * t))

def delayed_s_lambda(t, p):
    a, b = p[0], p[1]
    return a * (b**2) * t * np.exp(-b * t)

DelayedS = NHPPModel(
    param_names=['a', 'b'],
    m_func=delayed_s_m,
    lambda_func=delayed_s_lambda,
    initial_guess=[100.0, 0.05],
    bounds=[(1e-8, 1e6), (1e-12, 10)]
)

# 3. Inflection S-shaped Model
def inflection_s_m(t, p):
    a, b, beta = p[0], p[1], p[2]
    exp_term = np.exp(-b * t)
    return a * (1.0 - exp_term) / (1.0 + beta * exp_term)

def inflection_s_lambda(t, p):
    a, b, beta = p[0], p[1], p[2]
    exp_term = np.exp(-b * t)
    return a * b * (1.0 + beta) * exp_term / ((1.0 + beta * exp_term)**2)

InflectionS = NHPPModel(
    param_names=['a', 'b', 'beta'],
    m_func=inflection_s_m,
    lambda_func=inflection_s_lambda,
    initial_guess=[100.0, 0.05, 1.0],
    bounds=[(1e-8, 1e6), (1e-12, 10), (0.01, 100)]
)

# 4. Yamada Imperfect Debugging Type 1
def yamada_imperfect1_m(t, p):
    a, b, alpha = p[0], p[1], p[2]
    if alpha >= 1.0:
        alpha = 0.99
    return (a / (1.0 - alpha)) * (1.0 - np.exp(-b * (1.0 - alpha) * t))

def yamada_imperfect1_lambda(t, p):
    a, b, alpha = p[0], p[1], p[2]
    if alpha >= 1.0:
        alpha = 0.99
    return (a * b) * np.exp(-b * (1.0 - alpha) * t)

YamadaImperfect1 = NHPPModel(
    param_names=['a', 'b', 'alpha'],
    m_func=yamada_imperfect1_m,
    lambda_func=yamada_imperfect1_lambda,
    initial_guess=[100.0, 0.1, 0.1],
    bounds=[(1e-8, 1e6), (1e-12, 10), (0.0, 0.99)]
)

# 5. PNZ Model
def pnz_m(t, p):
    a, b, c, d = p[0], p[1], p[2], p[3]
    term1 = 1.0 - np.exp(-b * t)
    term2 = 1.0 - (c / (c + 1.0)) * np.exp(-d * t)
    return a * term1 * term2

def pnz_lambda(t, p):
    a, b, c, d = p[0], p[1], p[2], p[3]
    exp_bt = np.exp(-b * t)
    exp_dt = np.exp(-d * t)
    term1 = b * (1.0 - (c / (c + 1.0)) * exp_dt)
    term2 = (c * d / (c + 1.0)) * (1.0 - exp_bt)
    return a * (term1 + term2) * exp_bt

PNZ = NHPPModel(
    param_names=['a', 'b', 'c', 'd'],
    m_func=pnz_m,
    lambda_func=pnz_lambda,
    initial_guess=[100.0, 0.1, 1.0, 0.05],
    bounds=[(1e-8, 1e6), (1e-12, 10), (0.01, 100), (1e-12, 10)]
)

# 6. Vtub-shaped Model
def vtub_m(t, p):
    a, b, c = p[0], p[1], p[2]
    return a * (1.0 - np.exp(-b * t)) * (1.0 + c * t)

def vtub_lambda(t, p):
    a, b, c = p[0], p[1], p[2]
    exp_term = np.exp(-b * t)
    return a * b * exp_term * (1.0 + c * t) + a * c * (1.0 - exp_term)

Vtub = NHPPModel(
    param_names=['a', 'b', 'c'],
    m_func=vtub_m,
    lambda_func=vtub_lambda,
    initial_guess=[100.0, 0.1, 0.01],
    bounds=[(1e-8, 1e6), (1e-12, 10), (-1.0, 1.0)]
)

# Dictionary of all models
ALL_MODELS = {
    'Goel-Okumoto': GoelOkumoto,
    'Delayed-S': DelayedS,
    'Inflection-S': InflectionS,
    'Yamada-Imperfect-1': YamadaImperfect1,
    'PNZ': PNZ,
    'Vtub': Vtub
}

print(f"✅ Implemented {len(ALL_MODELS)} NHPP models!")
print("Models:", list(ALL_MODELS.keys()))

: 

In [ ]:
# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def aic(loglik, k):
    """Akaike Information Criterion"""
    return -2.0*loglik + 2.0*k

def bic(loglik, k, n):
    """Bayesian Information Criterion"""
    return -2.0*loglik + k*np.log(n)

def mse(y_obs, y_pred):
    """Mean Squared Error"""
    return np.mean((np.asarray(y_obs) - np.asarray(y_pred))**2)

def rmse(y_obs, y_pred):
    """Root Mean Squared Error"""
    return np.sqrt(mse(y_obs, y_pred))

def compare_models(failure_times, models_dict):
    """Compare multiple models"""
    results = []
    
    for name, model in models_dict.items():
        try:
            fit_res = model.fit_type2(failure_times)
            
            if fit_res['success']:
                n = len(failure_times)
                k = len(model.param_names)
                loglik = fit_res['loglik']
                
                cum_failures = np.arange(1, n+1)
                pred_failures = [model.predict_failures(t, fit_res['params']) for t in failure_times]
                
                results.append({
                    'Model': name,
                    'Params': k,
                    'LogLik': loglik,
                    'AIC': aic(loglik, k),
                    'BIC': bic(loglik, k, n),
                    'MSE': mse(cum_failures, pred_failures),
                    'RMSE': rmse(cum_failures, pred_failures),
                    'Parameters': fit_res['params']
                })
        except Exception as e:
            print(f"  ⚠️ {name} failed: {str(e)}")
    
    return pd.DataFrame(results)

def calculate_reliability(model, params, t, delta_t):
    """Calculate reliability R(delta_t | t)"""
    m_t = model.m_func(t, params)
    m_t_delta = model.m_func(t + delta_t, params)
    return np.exp(-(m_t_delta - m_t))

def calculate_mttf(model, params, t):
    """Calculate Mean Time To Failure"""
    intensity = model.lambda_func(t, params)
    return 1.0 / intensity if intensity > 0 else np.inf

print("✅ Utility functions defined!")

: 

<a id='data'></a>
## 3️⃣ Generate/Load Sample Data

Let's create synthetic failure data or load your own:

In [ ]:
# ============================================================================
# OPTION 1: Generate Synthetic Data
# ============================================================================

np.random.seed(42)

# Generate failure times from a Goel-Okumoto process
a_true, b_true = 80, 0.06
n_failures = 45

failure_times = []
for i in range(n_failures):
    # Generate interfailure time
    t_current = failure_times[-1] if failure_times else 0
    remaining_faults = a_true - len(failure_times)
    if remaining_faults > 0:
        mean_time = 1.0 / (b_true * remaining_faults)
        delta_t = np.random.exponential(mean_time)
        failure_times.append(t_current + delta_t)
    else:
        break

failure_times = np.array(failure_times)

print("📊 Synthetic Data Generated")
print(f"Number of failures: {len(failure_times)}")
print(f"Time range: {failure_times[0]:.2f} to {failure_times[-1]:.2f}")
print(f"Mean interfailure time: {np.mean(np.diff(failure_times)):.2f}")
print(f"\nFirst 10 failure times: {failure_times[:10]}")

# Save to CSV for future use
df = pd.DataFrame({'time': failure_times})
df.to_csv('failure_times.csv', index=False)
print("\n💾 Saved to 'failure_times.csv'")

: 

In [ ]:
# ============================================================================
# OPTION 2: Load Your Own Data (Uncomment to use)
# ============================================================================

# Upload your CSV file to Kaggle first, then:
# df = pd.read_csv('/kaggle/input/your-dataset/your_file.csv')
# failure_times = df['time'].values  # or whatever your column is named
# failure_times = np.sort(failure_times)

: 

<a id='explore'></a>
## 4️⃣ Exploratory Data Analysis

In [ ]:
# Basic statistics
n = len(failure_times)
T = failure_times[-1]
interfailure_times = np.diff(failure_times)

print("📈 DATA SUMMARY")
print("="*60)
print(f"Total failures observed:      {n}")
print(f"Observation period:           0 to {T:.2f}")
print(f"First failure time:           {failure_times[0]:.2f}")
print(f"Last failure time:            {failure_times[-1]:.2f}")
print(f"\nInterfailure Time Statistics:")
print(f"  Mean:                       {np.mean(interfailure_times):.2f}")
print(f"  Std Dev:                    {np.std(interfailure_times):.2f}")
print(f"  Min:                        {np.min(interfailure_times):.2f}")
print(f"  Max:                        {np.max(interfailure_times):.2f}")
print(f"\nFailure Rate:                 {n/T:.4f} failures/time unit")

: 

In [ ]:
# Laplace Trend Test
def laplace_trend_test(failure_times):
    """Test for trend in failure data"""
    n = len(failure_times)
    T = failure_times[-1]
    sum_ti = np.sum(failure_times)
    u = (sum_ti / n - T / 2) / (T / np.sqrt(12 * n))
    
    if u < -1.96:
        trend = "Reliability Growth (decreasing failure rate) ✅"
    elif u > 1.96:
        trend = "Reliability Degradation (increasing failure rate) ⚠️"
    else:
        trend = "Stable (constant failure rate) ➡️"
    
    return u, trend

u_stat, trend = laplace_trend_test(failure_times)

print("\n📊 LAPLACE TREND TEST")
print("="*60)
print(f"U-statistic:  {u_stat:.4f}")
print(f"Conclusion:   {trend}")
print("\nInterpretation:")
print("  U < -1.96  → Software reliability is improving")
print("  U > +1.96  → Software reliability is degrading")
print("  Otherwise  → Failure rate is stable")

: 

<a id='fit'></a>
## 5️⃣ Fit Individual Models

Let's fit each model one by one:

In [ ]:
# Fit Goel-Okumoto Model
print("🔧 Fitting Goel-Okumoto Model...")
result_go = GoelOkumoto.fit_type2(failure_times)

if result_go['success']:
    print("\n✅ Model fitted successfully!")
    print(f"\nParameters:")
    print(f"  a (total faults):          {result_go['params'][0]:.4f}")
    print(f"  b (detection rate):        {result_go['params'][1]:.6f}")
    print(f"\nGoodness of Fit:")
    print(f"  Log-Likelihood:            {result_go['loglik']:.4f}")
    print(f"  AIC:                       {aic(result_go['loglik'], 2):.4f}")
    print(f"  BIC:                       {bic(result_go['loglik'], 2, n):.4f}")
else:
    print(f"❌ Fitting failed: {result_go['message']}")

: 

In [ ]:
# Fit Delayed S-shaped Model
print("🔧 Fitting Delayed S-shaped Model...")
result_ds = DelayedS.fit_type2(failure_times)

if result_ds['success']:
    print("\n✅ Model fitted successfully!")
    print(f"\nParameters:")
    print(f"  a (total faults):          {result_ds['params'][0]:.4f}")
    print(f"  b (shape parameter):       {result_ds['params'][1]:.6f}")
    print(f"\nGoodness of Fit:")
    print(f"  Log-Likelihood:            {result_ds['loglik']:.4f}")
    print(f"  AIC:                       {aic(result_ds['loglik'], 2):.4f}")
    print(f"  BIC:                       {bic(result_ds['loglik'], 2, n):.4f}")

: 

<a id='compare'></a>
## 6️⃣ Compare All Models

Compare all implemented models at once:

In [ ]:
print("🔍 Comparing all models...\n")
results_df = compare_models(failure_times, ALL_MODELS)

# Sort by AIC (lower is better)
results_sorted = results_df.sort_values('AIC')

print("\n📊 MODEL COMPARISON RESULTS")
print("="*80)
print("\nRanked by AIC (lower is better):\n")
display(results_sorted[['Model', 'Params', 'LogLik', 'AIC', 'BIC', 'MSE', 'RMSE']])

best_model_name = results_sorted.iloc[0]['Model']
print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"   AIC = {results_sorted.iloc[0]['AIC']:.4f}")

: 

In [ ]:
# Show detailed parameters for best model
best_row = results_sorted.iloc[0]
best_model = ALL_MODELS[best_model_name]
best_params = best_row['Parameters']

print(f"\n📋 BEST MODEL DETAILS: {best_model_name}")
print("="*60)
print("\nFitted Parameters:")
for i, pname in enumerate(best_model.param_names):
    print(f"  {pname:15s} = {best_params[i]:.8f}")

print("\nModel Quality Metrics:")
print(f"  Log-Likelihood  = {best_row['LogLik']:.4f}")
print(f"  AIC             = {best_row['AIC']:.4f}")
print(f"  BIC             = {best_row['BIC']:.4f}")
print(f"  RMSE            = {best_row['RMSE']:.4f}")

: 

<a id='validate'></a>
## 7️⃣ Model Validation

Validate the best model:

In [ ]:
# Goodness-of-fit test (Kolmogorov-Smirnov)
def ks_test(failure_times, model, params):
    """Kolmogorov-Smirnov goodness-of-fit test"""
    n = len(failure_times)
    T = failure_times[-1]
    
    empirical_cdf = np.arange(1, n + 1) / n
    m_T = model.m_func(T, params)
    theoretical_cdf = np.array([model.m_func(t, params) / m_T for t in failure_times])
    
    D = np.max(np.abs(empirical_cdf - theoretical_cdf))
    critical_value_95 = 1.36 / np.sqrt(n)
    p_value = 1.0 if D < critical_value_95 else 0.05
    
    return D, p_value

D_stat, p_value = ks_test(failure_times, best_model, best_params)

print("\n🧪 GOODNESS-OF-FIT TEST")
print("="*60)
print("Kolmogorov-Smirnov Test:")
print(f"  D-statistic:    {D_stat:.6f}")
print(f"  P-value:        {p_value:.4f}")

if p_value > 0.05:
    print(f"  ✅ Model fits the data well (p > 0.05)")
else:
    print(f"  ⚠️ Model fit may be inadequate (p ≤ 0.05)")

: 

In [ ]:
# Residual Analysis
cum_failures = np.arange(1, n + 1)
predicted = [best_model.predict_failures(t, best_params) for t in failure_times]
residuals = cum_failures - np.array(predicted)

print("\n📊 RESIDUAL ANALYSIS")
print("="*60)
print(f"Mean residual:              {np.mean(residuals):.6f}")
print(f"Std deviation of residuals: {np.std(residuals):.6f}")
print(f"Max absolute residual:      {np.max(np.abs(residuals)):.6f}")

# Test for normality
if n > 8:
    _, p_norm = stats.shapiro(residuals)
    print(f"\nShapiro-Wilk normality test:")
    print(f"  P-value: {p_norm:.4f}")
    if p_norm > 0.05:
        print(f"  ✅ Residuals appear normally distributed")
    else:
        print(f"  ⚠️ Residuals may not be normally distributed")

: 

<a id='predict'></a>
## 8️⃣ Make Predictions

Use the best model for future predictions:

In [ ]:
# Current state
current_time = failure_times[-1]
current_failures = len(failure_times)
current_intensity = best_model.predict_intensity(current_time, best_params)

print("\n📍 CURRENT STATE")
print("="*60)
print(f"Current time:                 t = {current_time:.2f}")
print(f"Observed failures:            {current_failures}")
print(f"Predicted cumulative failures: {best_model.predict_failures(current_time, best_params):.2f}")
print(f"Current failure intensity:     λ(t) = {current_intensity:.6f}")
print(f"Current MTTF:                 {calculate_mttf(best_model, best_params, current_time):.2f}")

: 

In [ ]:
# Future predictions
future_times = [current_time + 10, current_time + 20, current_time + 50, current_time + 100]

print("\n🔮 FUTURE PREDICTIONS")
print("="*80)
print(f"{'Time':<12} {'Cumulative':<15} {'Additional':<15} {'Intensity':<15} {'MTTF':<12}")
print(f"{'(t)':<12} {'Failures':<15} {'Failures':<15} {'λ(t)':<15} {'1/λ(t)':<12}")
print("-"*80)

for ft in future_times:
    cum = best_model.predict_failures(ft, best_params)
    add = cum - current_failures
    intensity = best_model.predict_intensity(ft, best_params)
    mttf = 1.0/intensity if intensity > 0 else np.inf
    
    print(f"{ft:<12.2f} {cum:<15.2f} {add:<15.2f} {intensity:<15.6f} {mttf:<12.2f}")

: 

In [ ]:
# Reliability predictions
mission_times = [5, 10, 20, 50]

print("\n🛡️ RELIABILITY ANALYSIS")
print("="*60)
print(f"From current time t = {current_time:.2f}:\n")
print(f"{'Mission Duration':<20} {'Reliability R(Δt|t)':<20} {'Failure Prob':<20}")
print("-"*60)

for dt in mission_times:
    rel = calculate_reliability(best_model, best_params, current_time, dt)
    prob_fail = 1 - rel
    print(f"Δt = {dt:<15} {rel:<20.6f} {prob_fail:<20.6f}")

print("\nInterpretation:")
print("  R(Δt|t) = Probability of NO failures in next Δt time units")
print("  Higher reliability = Lower chance of failures")

: 

<a id='visualize'></a>
## 9️⃣ Visualizations

Create comprehensive plots:

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f'NHPP Software Reliability Analysis\nBest Model: {best_model_name}', 
             fontsize=16, fontweight='bold')

# Plot 1: Cumulative Failures
ax1 = axes[0, 0]
cum_failures = np.arange(1, n + 1)
t_plot = np.linspace(0, current_time * 1.3, 300)
m_plot = [best_model.predict_failures(t, best_params) for t in t_plot]

ax1.scatter(failure_times, cum_failures, color='red', s=50, label='Actual Failures', 
            zorder=5, alpha=0.7, edgecolors='darkred')
ax1.plot(t_plot, m_plot, 'b-', linewidth=3, label=f'{best_model_name} Fit')
ax1.set_xlabel('Time', fontsize=12, fontweight='bold')
ax1.set_ylabel('Cumulative Failures', fontsize=12, fontweight='bold')
ax1.set_title('Mean Value Function m(t)', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10, loc='lower right')
ax1.grid(True, alpha=0.3, linestyle='--')

# Plot 2: Failure Intensity
ax2 = axes[0, 1]
lambda_plot = [best_model.predict_intensity(t, best_params) for t in t_plot]
ax2.plot(t_plot, lambda_plot, 'g-', linewidth=3, label='Failure Intensity')
ax2.axhline(y=current_intensity, color='red', linestyle='--', linewidth=2, 
            label=f'Current: λ={current_intensity:.4f}')
ax2.set_xlabel('Time', fontsize=12, fontweight='bold')
ax2.set_ylabel('Failure Intensity λ(t)', fontsize=12, fontweight='bold')
ax2.set_title('Failure Intensity Function', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, linestyle='--')

# Plot 3: Residuals
ax3 = axes[1, 0]
ax3.scatter(failure_times, residuals, color='purple', s=40, alpha=0.6)
ax3.axhline(y=0, color='black', linestyle='--', linewidth=2)
ax3.axhline(y=np.mean(residuals), color='red', linestyle=':', linewidth=2, 
            label=f'Mean: {np.mean(residuals):.3f}')
ax3.fill_between(failure_times, -2*np.std(residuals), 2*np.std(residuals), 
                  alpha=0.2, color='gray', label='±2σ band')
ax3.set_xlabel('Time', fontsize=12, fontweight='bold')
ax3.set_ylabel('Residuals', fontsize=12, fontweight='bold')
ax3.set_title('Residual Plot', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3, linestyle='--')

# Plot 4: Interfailure Times
ax4 = axes[1, 1]
interfailure = np.diff(failure_times)
ax4.plot(range(1, len(interfailure) + 1), interfailure, 'o-', color='orange', 
         markersize=6, linewidth=2, alpha=0.7)
ax4.axhline(y=np.mean(interfailure), color='red', linestyle='--', linewidth=2,
            label=f'Mean: {np.mean(interfailure):.2f}')
ax4.set_xlabel('Failure Number', fontsize=12, fontweight='bold')
ax4.set_ylabel('Interfailure Time', fontsize=12, fontweight='bold')
ax4.set_title('Interfailure Time Sequence', fontsize=13, fontweight='bold')
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('nhpp_analysis.png', dpi=300, bbox_inches='tight')
print("\n📊 Plot saved as 'nhpp_analysis.png'")
plt.show()

: 

In [ ]:
# Model Comparison Chart
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

models_list = results_sorted['Model'].values
aic_values = results_sorted['AIC'].values

colors = ['green' if i == 0 else 'steelblue' for i in range(len(models_list))]
bars = ax.barh(models_list, aic_values, color=colors, edgecolor='black', linewidth=1.5)

# Add value labels
for i, (bar, val) in enumerate(zip(bars, aic_values)):
    ax.text(val + max(aic_values)*0.01, i, f'{val:.2f}', 
            va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('AIC (lower is better)', fontsize=12, fontweight='bold')
ax.set_title('Model Comparison by AIC', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x', linestyle='--')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
print("\n📊 Model comparison plot saved as 'model_comparison.png'")
plt.show()

: 

<a id='advanced'></a>
## 🔟 Advanced Analysis

Additional analyses and exports:

In [ ]:
# Generate prediction table
pred_times = np.linspace(0, current_time * 1.5, 100)
predictions_df = pd.DataFrame({
    'time': pred_times,
    'predicted_cumulative_failures': [best_model.predict_failures(t, best_params) for t in pred_times],
    'predicted_intensity': [best_model.predict_intensity(t, best_params) for t in pred_times],
    'mttf': [calculate_mttf(best_model, best_params, t) for t in pred_times]
})

print("\n📈 PREDICTION TABLE (first 10 rows):")
display(predictions_df.head(10))

# Save to CSV
predictions_df.to_csv('predictions.csv', index=False)
print("\n💾 Full predictions saved to 'predictions.csv'")

: 

In [ ]:
# Export comprehensive results
results_sorted.to_csv('model_comparison_results.csv', index=False)
print("💾 Model comparison results saved to 'model_comparison_results.csv'")

# Create summary report
summary = f"""
NHPP SOFTWARE RELIABILITY ANALYSIS SUMMARY
={'='*60}

DATASET INFORMATION
{'-'*60}
Number of failures observed:    {n}
Observation period:             0 to {T:.2f}
Mean interfailure time:         {np.mean(interfailure_times):.2f}

TREND ANALYSIS
{'-'*60}
Laplace U-statistic:            {u_stat:.4f}
Conclusion:                     {trend}

BEST MODEL
{'-'*60}
Model:                          {best_model_name}
Parameters:                     {', '.join([f'{p:.6f}' for p in best_params])}
AIC:                            {best_row['AIC']:.4f}
BIC:                            {best_row['BIC']:.4f}
RMSE:                           {best_row['RMSE']:.4f}

CURRENT STATE (t = {current_time:.2f})
{'-'*60}
Failure intensity λ(t):         {current_intensity:.6f}
MTTF:                           {calculate_mttf(best_model, best_params, current_time):.2f}

GOODNESS-OF-FIT
{'-'*60}
KS Test D-statistic:            {D_stat:.6f}
KS Test P-value:                {p_value:.4f}
Mean residual:                  {np.mean(residuals):.6f}
Std residual:                   {np.std(residuals):.6f}

Analysis completed successfully!
"""

print(summary)

with open('analysis_summary.txt', 'w') as f:
    f.write(summary)
print("\n💾 Summary saved to 'analysis_summary.txt'")

: 

## 🎉 Analysis Complete!

### 📦 Generated Files:
1. `failure_times.csv` - The dataset used
2. `predictions.csv` - Detailed predictions
3. `model_comparison_results.csv` - All model comparison metrics
4. `nhpp_analysis.png` - Comprehensive visualization
5. `model_comparison.png` - Model comparison chart
6. `analysis_summary.txt` - Text summary report

### 🔄 Next Steps:
- Upload your own failure data and rerun the analysis
- Modify the models to fit your specific use case
- Experiment with different model parameters
- Use predictions for project planning and resource allocation

### 📚 References:
- Pham, H. (2006). *System Software Reliability*. Springer.
- Chapter 6: NHPP Software Reliability Models

---

**Questions or Issues?**
- Check the USAGE_GUIDE.md for detailed instructions
- Review README.md for model descriptions
- All models are based on published research in software reliability engineering